# 03. CVRP: маршруты внутри кластера

Второй шаг схемы. На вход — группа точек и база, на выход — маршруты машин
с учётом вместимости и длительности смены.

Солвер — Google OR-Tools.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from routeforge.clustering import assign_to_depots, optimal_k_by_cluster_size
from routeforge.distance import haversine_matrix
from routeforge.io import read_points
from routeforge.solver import Fleet, SolverConfig, estimate_vehicles, solve_cvrp

sites = read_points("../data/sample/sites.csv")
depots_df = pd.read_csv("../data/sample/depots.csv")
depots = list(depots_df[["lat", "lon"]].itertuples(index=False, name=None))

# Берём один кластер, чтобы разобрать шаг подробно.
coords_all = list(sites[["lat", "lon"]].itertuples(index=False, name=None))
assigned = assign_to_depots(haversine_matrix(coords_all, depots))
block = sites[assigned == 0].reset_index(drop=True)

k, labels = optimal_k_by_cluster_size(
    np.asarray(list(block[["lat", "lon"]].itertuples(index=False, name=None))),
    max_per_cluster=40,
)
chunk = block[labels == 0].reset_index(drop=True)
print(f"кластер: {len(chunk)} точек, спрос {int(chunk.demand.sum()):,}")

## Подготовка задачи

Узел 0 — база, у неё нулевой спрос. Остальные узлы — точки обслуживания.

In [ ]:
points = [depots[0], *chunk[["lat", "lon"]].itertuples(index=False, name=None)]
matrix = haversine_matrix(points, points)
demands = [0, *(int(d) for d in chunk.demand)]

CAPACITY = 3000
n = estimate_vehicles(demands[1:], CAPACITY, matrix)
print(f"матрица {matrix.shape}, оценка числа машин: {n}")

`estimate_vehicles` берёт максимум из двух оценок — по суммарному спросу
и вместимости, и по суммарному времени работы. Занижать нельзя: солвер
просто не найдёт допустимого решения.

In [ ]:
fleet = Fleet(count=n, capacity=CAPACITY, max_time_min=480, speed_kmh=40, service_time_min=10)
solution = solve_cvrp(matrix, demands, fleet, depot=0, config=SolverConfig(time_limit_s=10))

print(f"маршрутов: {len(solution.routes)}, пропущено точек: {len(solution.dropped)}")
print(f"общий пробег: {solution.total_distance_m / 1000:.1f} км\n")
for r in solution.routes:
    print(f"  машина {r.vehicle}: {len(r.nodes) - 2:2} точек, "
          f"{r.distance_m / 1000:5.1f} км, {r.duration_min:3} мин, загрузка {r.load}/{CAPACITY}")

In [ ]:
plt.figure(figsize=(8, 6.5))
for i, route in enumerate(solution.routes):
    line = [points[j] for j in route.nodes]
    plt.plot([p[1] for p in line], [p[0] for p in line], lw=1.3, alpha=0.85,
             label=f"машина {route.vehicle} ({route.load})")
plt.scatter([p[1] for p in points[1:]], [p[0] for p in points[1:]], s=16, c="0.5", zorder=3)
plt.scatter(points[0][1], points[0][0], s=220, c="k", marker="*", zorder=5)
plt.title(f"{len(solution.routes)} маршрута, {solution.total_distance_m / 1000:.1f} км")
plt.legend(fontsize=8)
plt.grid(alpha=0.2)

## Влияние вместимости

Чем меньше машина, тем больше маршрутов и тем длиннее суммарный пробег:
каждый лишний рейс это лишний заезд на базу и обратно.

In [ ]:
rows = []
for cap in (1500, 2000, 3000, 4000, 6000):
    m = estimate_vehicles(demands[1:], cap, matrix)
    s = solve_cvrp(matrix, demands, Fleet(count=m, capacity=cap), depot=0,
                   config=SolverConfig(time_limit_s=5))
    rows.append({"вместимость": cap, "машин": len(s.routes),
                 "пробег, км": round(s.total_distance_m / 1000, 1),
                 "пропущено": len(s.dropped)})
pd.DataFrame(rows)

## Пропуск точек

Если машин не хватает, солвер может пропустить точку, заплатив `drop_penalty`.
Штраф должен заметно превышать стоимость крюка до самой дальней точки, иначе
бросать станет выгоднее, чем заезжать.

Когда пропуск недопустим в принципе, ставится `drop_penalty=None` — тогда при
нехватке машин вернётся пустое решение вместо частичного. Это честнее, чем
молча потерять точки.

In [ ]:
# Одна машина на 800 единиц при спросе больше 2000 — увезти всё физически нельзя.
short = Fleet(count=1, capacity=800)

with_drop = solve_cvrp(matrix, demands, short, depot=0,
                       config=SolverConfig(time_limit_s=5))
without   = solve_cvrp(matrix, demands, short, depot=0,
                       config=SolverConfig(time_limit_s=5, drop_penalty=None))

print(f"спрос кластера: {sum(demands)}, вместимость машины: 800\n")
print(f"со штрафом за пропуск: {len(with_drop.routes)} маршрут, "
      f"обслужено {len(with_drop.routes[0].nodes) - 2} точек, "
      f"пропущено {len(with_drop.dropped)}")
print(f"пропуск запрещён:      маршрутов {len(without.routes)} "
      f"— допустимого решения нет, солвер вернул пустой результат")

## Раздельные точки старта и финиша

Машина не всегда возвращается туда, откуда выехала: типичный случай — выезд
из гаража, выгрузка на полигоне. Для этого вместо `depot` задаются `starts`
и `ends`.

In [ ]:
garage = (55.20, 61.30)
landfill = depots[0]

pts = [garage, landfill, *chunk[["lat", "lon"]].itertuples(index=False, name=None)]
m2 = haversine_matrix(pts, pts)
dem2 = [0, 0, *(int(d) for d in chunk.demand)]

sol2 = solve_cvrp(m2, dem2, Fleet(count=n, capacity=CAPACITY),
                  starts=[0] * n, ends=[1] * n, config=SolverConfig(time_limit_s=10))

for r in sol2.routes:
    print(f"машина {r.vehicle}: старт узел {r.nodes[0]}, финиш узел {r.nodes[-1]}, "
          f"{r.distance_m / 1000:.1f} км")

Весь пайплайн целиком — `routeforge.pipeline.plan_routes_sync`,
пример в [README](../README.md).